In [1]:
import sys
sys.path.append("../")
from chess_engine.src.model.classes.sqlite.models import (GamePositionRollup)
from chess_engine.src.model.classes.sqlite.database import SessionLocal
from chess_engine.src.model.classes.bitboard_processing.bitboard_creator import get_all_bitboards_dict
from chess_engine.src.model.config.config import Settings
from typing import List, Tuple
from sqlalchemy import or_, and_, func
import chess
import re
import pandas as pd
import numpy as np
import json 
from tqdm import tqdm
from sqlalchemy import or_, and_, func
from chess_engine.src.model.classes.bitboard_processing.bitboard_creator import bitboards_to_array, sample_bitboard_dict, Bitboard_Creator
from chess_engine.src.model.classes.sqlite.database import  get_db
from chess_engine.src.model.classes.sqlite.models import GamePositionRollup
from chess_engine.src.model.classes.bitboard_processing.bitboard_creator import bitboards_to_array, sample_bitboard_dict, Bitboard_Creator
from chess_engine.src.model.classes.sqlite.database import  get_db
import numpy as np
from torch.utils.data import Dataset, DataLoader
import torch
import os
# src\model\classes\npz_piping\create_npz_files.py
from chess_engine.src.model.classes.npz_piping.create_npz_files import db_to_npz_files
from chess_engine.src.model.config.config import settings

In [2]:
db_to_npz_files()

Deleted: ./src/model/data/training\data_0.npz
Deleted: ./src/model/data/training\data_1.npz
Deleted: ./src/model/data/training\data_10.npz
Deleted: ./src/model/data/training\data_11.npz
Deleted: ./src/model/data/training\data_12.npz
Deleted: ./src/model/data/training\data_13.npz
Deleted: ./src/model/data/training\data_14.npz
Deleted: ./src/model/data/training\data_15.npz
Deleted: ./src/model/data/training\data_16.npz
Deleted: ./src/model/data/training\data_17.npz
Deleted: ./src/model/data/training\data_18.npz
Deleted: ./src/model/data/training\data_19.npz
Deleted: ./src/model/data/training\data_2.npz
Deleted: ./src/model/data/training\data_20.npz
Deleted: ./src/model/data/training\data_21.npz
Deleted: ./src/model/data/training\data_22.npz
Deleted: ./src/model/data/training\data_23.npz
Deleted: ./src/model/data/training\data_24.npz
Deleted: ./src/model/data/training\data_25.npz
Deleted: ./src/model/data/training\data_26.npz
Deleted: ./src/model/data/training\data_27.npz
Deleted: ./src/m

In [31]:
class NPZDataset(Dataset):
    def __init__(self, directory):
        self.npz_files = []
        self.file_offsets = []
        self.total_samples = 0

        all_entries = os.listdir(directory)
        # print(f"Found {len(all_entries)} entries in {directory}.")  # Debugging print

        for entry in all_entries:
            file_path = os.path.join(directory, entry)
            if os.path.isfile(file_path) and entry.endswith('.npz'):
                # print(f"Processing file: {file_path}")  # Debugging print
                data = np.load(file_path)
                n_samples = data['features'].shape[0]
                # print(f"File {file_path} has {n_samples} samples.")  # Debugging print
                self.npz_files.append(file_path)
                self.file_offsets.append(self.total_samples)
                self.total_samples += n_samples

        # print(f"Total samples in dataset: {self.total_samples}")  # Debugging print

    def __len__(self):
        return self.total_samples

    def __getitem__(self, idx):
        try:
            # print(f"Fetching index: {idx}")  # Debugging print
            file_idx = next(i for i, offset in enumerate(self.file_offsets) if idx < offset)
            if file_idx > 0:
                idx -= self.file_offsets[file_idx - 1]
    
            # print(f"Index {idx} maps to file {self.npz_files[file_idx]}")  # Debugging print
    
            data = np.load(self.npz_files[file_idx])
            features = data['features'][idx]
            labels = data['labels'][idx]
    
            # print(f"Features shape: {features.shape}, Labels shape: {labels.shape}")  # Debugging print
            
            return torch.tensor(features, dtype=torch.float32), torch.tensor(labels, dtype=torch.long)
        except Exception as e:
            print(f"Error fetching index {idx}: {e}")
            raise  # Re-raise the exception to halt execution in the second block
    


In [17]:
dataset = NPZDataset(settings.npzTrainingDirectory)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True, num_workers=0)

print(f"Dataset length: {len(dataset)}")


Dataset length: 14479


In [11]:
sample_features, sample_labels = dataset[0]
print(f"Sample 0: Features shape {sample_features.shape}, Labels {sample_labels}")

Fetching index: 0
Features shape (12, 8, 8), Labels shape (3,)
Sample 0: Features shape torch.Size([12, 8, 8]), Labels tensor([0., 1., 0.])


In [36]:

for i, (features, labels) in enumerate(dataloader):
    print(f"Batch {i+1}: Features shape {features.shape}, Labels shape {labels.shape}")
    # if i == 5:  # Stop after 5 batches to limit output
    #     break
    



Fetching index: 2420
Features shape (12, 8, 8), Labels shape (3,)
Fetching index: 9269
Features shape (12, 8, 8), Labels shape (3,)
Fetching index: 5900
Features shape (12, 8, 8), Labels shape (3,)
Fetching index: 1344
Features shape (12, 8, 8), Labels shape (3,)
Fetching index: 13572
Features shape (12, 8, 8), Labels shape (3,)
Fetching index: 11918
Features shape (12, 8, 8), Labels shape (3,)
Fetching index: 9542
Features shape (12, 8, 8), Labels shape (3,)
Fetching index: 7319
Features shape (12, 8, 8), Labels shape (3,)
Fetching index: 5147
Features shape (12, 8, 8), Labels shape (3,)
Fetching index: 9466
Features shape (12, 8, 8), Labels shape (3,)
Fetching index: 925
Features shape (12, 8, 8), Labels shape (3,)
Fetching index: 690
Features shape (12, 8, 8), Labels shape (3,)
Fetching index: 6534
Features shape (12, 8, 8), Labels shape (3,)
Fetching index: 13277
Features shape (12, 8, 8), Labels shape (3,)
Fetching index: 2498
Features shape (12, 8, 8), Labels shape (3,)
Fetching 

In [33]:
try:
    for features, labels in dataloader:
        print(f"Batch features shape: {features.shape}")
        print(f"Batch labels shape: {labels.shape}")
except Exception as e:
    print(f"Error during DataLoader iteration: {e}")


Fetching index: 1520
Features shape (12, 8, 8), Labels shape (3,)
Fetching index: 4476
Features shape (12, 8, 8), Labels shape (3,)
Fetching index: 5753
Features shape (12, 8, 8), Labels shape (3,)
Fetching index: 7788
Features shape (12, 8, 8), Labels shape (3,)
Fetching index: 8980
Features shape (12, 8, 8), Labels shape (3,)
Fetching index: 1840
Features shape (12, 8, 8), Labels shape (3,)
Fetching index: 14401


In [29]:

data0 = np.load('./src/model/data/training/data_0.npz')
data1 = np.load('./src/model/data/training/data_19.npz')

features0 = data0['features']
features1 = data1['features']


datas = [features0,features1]

In [32]:
features1[0].shape

(12, 8, 8)

In [21]:
d = np.concatenate(datas, axis=0)

In [20]:
for i, v in enumerate([0,500,1000]):
    print(i,v)

0 0
1 500
2 1000


In [26]:
dataset = NPZDataset(settings.npzTrainingDirectory)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True, num_workers=0)

In [27]:
import torch
import torch.nn as nn
import torch.optim as optim

class SampleModel(nn.Module):
    def __init__(self, input_size=12*8*8, num_classes=3):
        super(SampleModel, self).__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(input_size, 128)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128, num_classes)
        self.softmax = nn.Softmax(dim=1)  # For multi-class classification

    def forward(self, x):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return self.softmax(x)


In [28]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for features, labels in dataloader:
        features, labels = features.to(device), labels.to(device)

        # One-hot to class index if labels are one-hot encoded
        labels = torch.argmax(labels, dim=1)

        # Forward pass
        outputs = model(features)
        loss = criterion(outputs, labels)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

    accuracy = 100 * correct / total
    print(f"Train Loss: {total_loss:.4f}, Train Accuracy: {accuracy:.2f}%")
    return total_loss, accuracy


def evaluate_model(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for features, labels in dataloader:
            features, labels = features.to(device), labels.to(device)

            # One-hot to class index if labels are one-hot encoded
            labels = torch.argmax(labels, dim=1)

            # Forward pass
            outputs = model(features)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)

    accuracy = 100 * correct / total
    print(f"Validation/Test Loss: {total_loss:.4f}, Accuracy: {accuracy:.2f}%")
    return total_loss, accuracy


In [41]:
# Import necessary libraries
import torch
from torch.utils.data import DataLoader
from torch.nn import CrossEntropyLoss
from torch.optim import Adam

# Assume you already have train_loader, val_loader, and test_loader
# These should be DataLoader objects from your Dataset class

# Hyperparameters
input_size = 12 * 8 * 8
num_classes = 3
learning_rate = 0.001
epochs = 5

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Initialize the model, criterion, and optimizer
model = SampleModel(input_size=input_size, num_classes=num_classes).to(device)
criterion = CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=learning_rate)

train_dataset = NPZDataset(settings.npzTrainingDirectory)
test_dataset = NPZDataset(settings.npzTestingDirectory)
val_dataset = NPZDataset(settings.npzValidationDirectory)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=True, num_workers=0)

# Train and validate the model
for epoch in range(epochs):
    print(f"Epoch {epoch+1}/{epochs}")
    train_one_epoch(model, train_loader, criterion, optimizer, device)
    evaluate_model(model, val_loader, criterion, device)

# Test the model on the test set
print("\nTesting on Test Dataset:")
evaluate_model(model, test_loader, criterion, device)


Epoch 1/5
Error fetching index 14361: 
Train Loss: 1.0962, Train Accuracy: 43.75%
Error fetching index 76: 


ZeroDivisionError: division by zero